## Environment setup

In [ ]:
from datetime import datetime 
import importlib
from functools import partial

import tensorflow as tf
from tensorflow.data import Dataset, TFRecordDataset
import tensorflow_datasets as tfds

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

# Notebooks run from VSCode use home directory as a base path
# while notebooks ran from JupyterLab use the current directory as a base path
import sys, os

home = os.getenv("HOME")
if home == "/home/jovyan":
    sys.path.append("/home/jovyan/ELITPC/TPCReco/PythonAnalysis/python/")
else:
    sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")
    os.chdir('/home/akalinow/scratch/ELITPC/PythonAnalysis/')

os.makedirs("fig_png", exist_ok=True)

#Increase plots font size
params = {'legend.fontsize': 'xx-large',
          'figure.figsize': (10, 7),
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

2026-03-30 08:28:46.647149: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 08:28:46.681077: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-30 08:28:47.900239: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Load the dataset and convert to pandas DataFrame

In [ ]:
# ROOT file written by tpcWriter with --output.mlTree true
fileName = 'SimEvent_Track3D_TwoProng_gun_MC_100k.root'

import mltree_tfrecord as mt
importlib.reload(mt)

# ROOT file written by tpcWriter with the MLData tree (--output.mlTree true)
dataPath = '/home/jovyan/ELITPC/TPCReco/build/resources/'

# Convert the MLData tree to TFRecord files (once per dataset)
tfrecordDir = 'TFRecord_TwoProng_gun_MC_test'
nEvents, shards = mt.convert_file(dataPath+fileName, tfrecordDir)
print(nEvents, "events in", len(shards), "TFRecord files")

# SimEvent and RecoEvent targets in one dataset: {'sim': (image, target), 'reco': (image, target)}
mergedDataset = mt.load_merged_dataset(tfrecordDir+'/*.tfrecord.gz')

# select events with the simulated alpha track length in [minAlphaLength, maxAlphaLength] mm
minAlphaLength = 30.0
maxAlphaLength = 10000.0
mergedDataset = mergedDataset.filter(
    lambda x: (mt.segment_length(x['sim'][1], 0) > minAlphaLength) &
              (mt.segment_length(x['sim'][1], 0) < maxAlphaLength)
)

In [ ]:
%%time 

import utility_functions as utils
importlib.reload(utils)

dataset = mergedDataset  # from the cell above

data = dataset.map(lambda x: tf.reshape(x["sim"][1], (1,-1))).unbatch().as_numpy_iterator()
df = pd.DataFrame(data, columns=utils.columns_XYZ)

data = dataset.map(lambda x: tf.reshape(x["reco"][1], (1,-1))).unbatch().as_numpy_iterator()
df_reco = pd.DataFrame(data, columns=utils.columns_XYZ)

df = df.merge(df_reco, left_index=True, right_index=True, suffixes=("_sim", "_reco"), how="outer");

### Load the model


In [ ]:
import plotting_functions as plf
importlib.reload(plf)

path = '/home/jovyan/ELITPC/PythonAnalysis/training/0100_2026_Mar_27_17_45_38.keras'
model = tf.keras.models.load_model(path)

#dataset_f = dataset.map(lambda x: (tf.where(x['sim'][0]>0.1, x['sim'][0], 0), x['sim'][1]))

x = iter(dataset.batch(1))
for i in range(1):
    item = next(x)
    plf.plotEvent(item['sim'], model=model)

### Resolution plots in XYZ

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
plf.plotEndPointRes(df=df, edge="Vtx", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df, edge="Alpha", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df, edge="Carbon", coordinates=["x", "y", "z"])

plf.plotLengthPull(df, partName="Alpha")
plf.plotLengthPull(df, partName="Carbon")
plf.plotLengthPullEvolution(df)
plf.plotOpeningAngleCos(df)

### Resolution plots for filtered dataset

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

partName = "Carbon"
d_reco = np.sqrt((df["x"+partName+"_reco"] - df["xVtx_reco"])**2 + 
                     (df["y"+partName+"_reco"] - df["yVtx_reco"])**2 + 
                     (df["z"+partName+"_reco"] - df["zVtx_reco"])**2 )


partName = "Alpha"
d_sim = np.sqrt((df["x"+partName+"_sim"] - df["xVtx_sim"])**2 + 
                     (df["y"+partName+"_sim"] - df["yVtx_sim"])**2 + 
                     (df["z"+partName+"_sim"] - df["zVtx_sim"])**2 )

#mask = (df["pullAlpha"]<-10)*(np.abs(df["xAlpha_sim"])<100)*(np.abs(df["yAlpha_sim"])<90)*(np.abs(df["zAlpha_sim"])<50)
#mask = np.abs(d_reco-15)>0
mask = (d_sim<100)*(d_sim>30)

plf.controlPlots(df[mask])
plf.plotEndPointRes(df=df[mask], edge="Vtx", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df[mask], edge="Alpha", coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df[mask], edge="Carbon", coordinates=["x", "y", "z"])

plf.plotLengthPull(df[mask], partName="Alpha")
plf.plotLengthPull(df[mask], partName="Carbon")
plf.plotLengthPullEvolution(df[mask])

### Resolution plots in UVWT

In [ ]:
import utility_functions as utils
importlib.reload(utils)

batchSize = 32
import mltree_tfrecord as mt
dataset = mt.load_merged_dataset('TFRecord_TwoProng_gun_MC/*.tfrecord.gz')

data = dataset.map(lambda x: tf.reshape(x["sim"][1], (1,-1)))
data = data.map(lambda x: tf.reshape(x, (-1,3,3)))
data = data.map(lambda x: utils.XYZtoUVWT_event(x))
data = data.map(lambda x: tf.reshape(x, (-1,12))).unbatch().as_numpy_iterator()

df = pd.DataFrame(data, columns=utils.columns_UVWT)

data = dataset.map(lambda x: tf.reshape(x["reco"][1], (1,-1)))
data = data.map(lambda x: tf.reshape(x, (-1,3,3)))
data = data.map(lambda x: utils.XYZtoUVWT_event(x))
data = data.map(lambda x: tf.reshape(x, (-1,12))).unbatch().as_numpy_iterator()
df_reco = pd.DataFrame(data, columns=utils.columns_UVWT)

df = df.merge(df_reco, left_index=True, right_index=True, suffixes=("_sim", "_reco"), how="outer");

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

plf.plotEndPointRes(df=df, edge="Vtx", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Alpha", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Carbon", coordinates=["u", "v", "w", "t"])